# Packr engine demo

The same engine as the earlier demo notebook, now imported from the `packr` package. Run `pip install -e .` from the repo root first.
Material values and rule thresholds are placeholders (see `ROADMAP.md`), so read these results as a demonstration of the logic, not as advice.

In [1]:
import json
import pandas as pd

from packr.engine import Scenario, recommend, format_report
from packr.materials import load_materials, required_thickness_um
from packr.paths import REFERENCE_DIR

## Dry fatty snack

In [2]:
chips = Scenario("Potato Chips", moisture_pct=2, fat_pct=33, ph=6.0, shelf_life_days=180, temp_c=25, rh_pct=60)
print(format_report(recommend(chips)))

INPUT: Potato Chips | moisture 2% | fat 33% | pH 6 | respiration None
       shelf life 180 d | 25 C | RH 60% | Ambient | transport Normal

WHY (Level 1 risks):
  * Fat 33% -> oxidation risk High
  * Moisture 2% with RH 60% -> moisture gain risk High
  * Fat-rich product is light sensitive -> needs a light barrier

WHY (requirement rules):
  * Oxidation High + long (>90 d) shelf life -> OTR <= 5
  * Moisture gain High + long (>90 d) shelf life -> WVTR <= 1
  * Transport Normal -> mechanical strength >= 3/5

RECOMMENDED PACKAGING: 2 layer(s)   (25 of 89 candidates passed)
  layer 1 (outer): MetPET
  layer 2 (sealant): CPP

REQUIRED PROPERTIES:
  OTR                 -> Low (<= 5 cc/m2/day)
  WVTR                -> Very Low (<= 1 g/m2/day)
  Light barrier       -> High
  Sealability         -> High
  Mechanical strength -> Moderate
  MAP                 -> Recommended: nitrogen flushing to cut residual oxygen

ACHIEVED BY THIS STRUCTURE:
  OTR 1.5 cc/m2/day | WVTR 0.889 g/m2/day | light 0

## Fresh produce, using the respiration rate from the commodity seed table

The numeric rate overrides the High/Medium/Low label. It must be a rate at the storage temperature (see `units.q10_adjust`).

In [3]:
seed = pd.read_csv(REFERENCE_DIR / "commodities_produce_seed.csv")
rate = float(seed.loc[seed["commodity"] == "Strawberry", "respiration_rate_mgco2_kg_hr"].iloc[0])
strawberry = Scenario("Strawberries", moisture_pct=91, fat_pct=0.3, ph=3.5, shelf_life_days=7, temp_c=5,
                      rh_pct=90, storage="Chilled", respiration_mgco2_kg_h=rate)
print(f"seed respiration rate: {rate:g} mg CO2/kg/h")
print(format_report(recommend(strawberry)))

seed respiration rate: 30 mg CO2/kg/h
INPUT: Strawberries | moisture 91% | fat 0.3% | pH 3.5 | respiration 30 mg CO2/kg/h
       shelf life 7 d | 5 C | RH 90% | Chilled | transport Normal

WHY (Level 1 risks):
  * Fat 0.3% -> oxidation risk None
  * Moisture 91% -> dehydration risk Medium
  * Moisture 91% and pH 3.5 -> microbial risk Low
  * Respiration High -> film must let O2/CO2 exchange

WHY (requirement rules):
  * Respiration High -> OTR >= 2000 (avoid anaerobic conditions)
  * Dehydration risk -> WVTR <= 30
  * Transport Normal -> mechanical strength >= 3/5
  * High respiration -> prefer micro-perforated / breathable film

RECOMMENDED PACKAGING: 1 layer(s)   (2 of 89 candidates passed)
  layer 1 (monolayer): MicroperfBOPP

REQUIRED PROPERTIES:
  OTR                 -> At least 2000 cc/m2/day (breathable)
  WVTR                -> High (<= 30 g/m2/day)
  Light barrier       -> Not critical
  Sealability         -> Moderate
  Mechanical strength -> Moderate
  MAP                 ->

## Inputs are validated

In [4]:
try:
    Scenario("Impossible", moisture_pct=70, fat_pct=40, ph=6, shelf_life_days=30, temp_c=20, rh_pct=50)
except ValueError as err:
    print("rejected:", err)

rejected: moisture_pct + fat_pct cannot exceed 100


## Thickness comes from permeability

Bulk polymers store an intrinsic permeability, so the gauge at which a film exactly meets an OTR limit can be computed. It is a signal, not a spec: the table does not yet hold minimum or maximum practical gauges. Coated, metallised and foil layers are fixed-barrier and cannot be tuned this way.

In [5]:
mats = load_materials()
limit = 1000
for name in ("LDPE", "BOPA", "EVOH"):
    need, nominal = required_thickness_um(mats, name, limit), mats.loc[name, "gauge_um"]
    verdict = (f"needs {need:.0f} um, {need / nominal:.1f}x its nominal {nominal:g} um: impractical on its own"
               if need > nominal else
               f"already meets the limit at its nominal {nominal:g} um (the limit alone would allow {need:.3g} um)")
    print(f"{name:5s} OTR <= {limit}: {verdict}")
try:
    required_thickness_um(mats, "MetPET", limit)
except ValueError as err:
    print("MetPET:", err)

LDPE  OTR <= 1000: needs 325 um, 6.5x its nominal 50 um: impractical on its own
BOPA  OTR <= 1000: already meets the limit at its nominal 15 um (the limit alone would allow 0.6 um)
EVOH  OTR <= 1000: already meets the limit at its nominal 5 um (the limit alone would allow 0.005 um)
MetPET: MetPET is a fixed-barrier layer; its OTR does not scale with thickness


## Structured output, the shape the ML stage will train on

In [6]:
rec = recommend(chips).to_dict()
print(json.dumps({k: rec[k] for k in ("n_scanned", "n_passing", "notes")}, indent=2))
print(json.dumps(rec["top"][0], indent=2))

{
  "n_scanned": 89,
  "n_passing": 25,
  "notes": [
    "Material barrier values are placeholders, not datasheet or literature values."
  ]
}
{
  "layers": [
    "MetPET",
    "CPP"
  ],
  "OTR": 1.4992503748125938,
  "WVTR": 0.8888888813162806,
  "light": 0.95,
  "seal": 5.0,
  "strength": 4.0,
  "cost": 4.0,
  "thickness_um": 37.0,
  "recyclable": false,
  "breathable": false
}
